# Matched observation action and window adapter

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import numpy as np
import gymnasium as gym
from preference_scoring import FrozenPreferenceScore,estimator_state_hash
from baseline_rewards import TaskReward,MaxSupportReward
SCHEMA='solid81_time_qincrease_valid_fresh_age_v1'
class BaselineState:
    """Decision-local state shared by TASK/MAX; no native reward-function calls."""
    def __init__(self, score_model, condition, horizon=600, window_decisions=15):
        if condition not in ('TASK', 'MAX'):
            raise ValueError('Only TASK and MAX belong to this adapter')
        if horizon != 600 or window_decisions != 15:
            raise ValueError('This audited development contract fixes horizon600/window15')
        self.model, self.condition = score_model, condition
        self.horizon, self.window_decisions = horizon, window_decisions
        self.task_reward=TaskReward(); self.max_reward=MaxSupportReward()
        self.episode = 0
        self.needs_reset = True

    def reset(self, game_obs):
        self.episode += 1
        self.task_reward.reset(); self.max_reward.reset()
        self.decision, self.high_score = 0, 0.0
        self.rows, self.previous_window = [], None
        self.q, self.valid, self.last_update = 0.0, False, None
        self.window_id = 0
        self.needs_reset = False
        return self.observation(game_obs, False)

    def observation(self, game_obs, fresh):
        game_obs = np.asarray(game_obs, dtype=np.float32)
        if game_obs.shape != (81,) or not np.isfinite(game_obs).all():
            raise ValueError('Expected finite 81-field raw game observation')
        age = 0 if self.last_update is None else self.decision - self.last_update
        # time/horizon = decision/600 under the pinned fixed-step clock.
        # Physical seconds use the pinned player timing contract.
        tail = [self.decision / self.horizon, self.q, float(self.valid), float(fresh), age / self.horizon]
        return np.r_[game_obs, np.asarray(tail, dtype=np.float32)].astype(np.float32)

    def step(self, game_obs, raw_score, terminated=False, truncated=False):
        if self.needs_reset:
            raise RuntimeError('reset() required before stepping or after episode ending')
        game_obs = np.asarray(game_obs, dtype=np.float32)
        if game_obs.shape != (81,) or not np.isfinite(game_obs).all() or not np.isfinite(raw_score):
            raise ValueError('Invalid telemetry; no clipping or nan_to_num repair')
        self.decision += 1
        if self.decision > self.horizon:
            raise RuntimeError('Decision exceeds frozen horizon')
        before = self.high_score
        self.high_score = max(before, float(raw_score))
        b = self.task_reward.calculate(float(raw_score))  # identity normalization; one per score-increase event
        self.rows.append(game_obs[-27:].astype(np.float64))
        fresh, pair, diagnostic = False, None, None
        if len(self.rows) == self.window_decisions:
            current = np.mean(self.rows, axis=0)
            self.window_id += 1
            if self.previous_window is not None:
                pair = np.r_[self.previous_window, current]
                self.q, diagnostic = self.model(pair)
                if not np.isfinite(self.q) or not 0 <= self.q <= 1:
                    raise ValueError('Invalid score output')
                self.valid, fresh, self.last_update = True, True, self.decision
            self.previous_window = current
            self.rows = []
        affect = self.max_reward.calculate(self.q,self.valid,fresh,[self.episode,self.window_id-1,self.window_id] if fresh else None)
        reward = b if self.condition == 'TASK' else affect
        if self.decision == self.horizon and not terminated:
            truncated = True
        self.needs_reset = bool(terminated or truncated)
        info = {'episode': self.episode, 'decision': self.decision, 'condition': self.condition,
                'raw_score_signal': float(raw_score), 'score_high_water_before': before,
                'score_high_water': self.high_score, 'b': b, 'task_normalization': 'identity',
                'q': self.q if self.valid else None, 'valid': self.valid, 'fresh': fresh,
                'age_decisions': None if self.last_update is None else self.decision - self.last_update,
                'completed_window_id': self.window_id,
                'pair_id': [self.episode, self.window_id - 1, self.window_id] if fresh else None,
                'pair_input': pair.tolist() if pair is not None else None, 'estimator': diagnostic,
                'affect_impulse': affect, 'delivered_reward': reward,
                'derived_game_seconds': self.decision * 0.2,
                'clock_status': 'derived_fixed_step_from_pinned_player',
                'comparison_decision': self.decision if fresh else None,
                'availability_decision': self.decision if fresh else None,
                'comparison_game_seconds': self.decision * 0.2 if fresh else None,
                'availability_game_seconds': self.decision * 0.2 if fresh else None,
                'pair_window_intervals_seconds': [[(self.window_id-2)*3.0,(self.window_id-1)*3.0],[(self.window_id-1)*3.0,self.window_id*3.0]] if fresh else None,
                'terminated': bool(terminated), 'truncated': bool(truncated)}
        return self.observation(game_obs, fresh), reward, bool(terminated), bool(truncated), info

class MatchedBaseline(gym.Wrapper):
    """Common interface; native output retained in info for paired characterization."""
    def __init__(self, env, condition):
        super().__init__(env)
        native = env.env
        expected = (native.game == 'Solid' and native.classifier and native.preference
                    and not native.imitation_learning and not native.discretize
                    and native.decision_period == 10 and native.cluster == 0
                    and not native.period_ra and native.weight == 0.5)
        if not expected or tuple(env.observation_space.shape) != (86,):
            raise ValueError('Native configuration differs from audited Solid vector route')
        self.score_model = FrozenPreferenceScore(native.model)
        self.state = BaselineState(self.score_model, condition)
        self.action_space = gym.spaces.MultiDiscrete([3, 3])
        low = np.r_[np.full(81, -np.inf), np.zeros(5)].astype(np.float32)
        high = np.r_[np.full(81, np.inf), np.ones(5)].astype(np.float32)
        self.observation_space = gym.spaces.Box(low, high, dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        # Seed this Python wrapper only. Unity receives its distinct initialization
        # seed in the factory handshake; RESET protocol carries no seed field.
        if seed is not None:
            self.np_random, _ = gym.utils.seeding.np_random(seed)
        native_obs, _ = self.env.reset(options=options)
        self.score_model.assert_frozen()
        obs = self.state.reset(np.asarray(native_obs)[:81])
        return obs, {'schema': SCHEMA, 'simulator_initialization_seed': getattr(self.env.env, 'simulator_initialization_seed', None),
                     'wrapper_reset_seed': seed, 'reset_reseeds_unity': False,
                     'clock_status': 'derived_fixed_step_from_pinned_player'}

    def step(self, action):
        if not self.action_space.contains(action):
            raise ValueError('Illegal steering/pedal action')
        if self.state.needs_reset:
            raise RuntimeError('reset required')
        native_action = np.r_[np.asarray(action, dtype=np.int64), 0]
        native_obs, native_reward, terminated, truncated, native_info = self.env.step(native_action)
        raw = native_info['step']
        if len(raw.reward) != 1:
            raise ValueError('Expected one Unity agent')
        raw_score = float(raw.reward[0])
        result = self.state.step(np.asarray(native_obs)[:81], raw_score, terminated, truncated)
        obs, reward, terminated, truncated, info = result
        info.update({'native_action': native_action.tolist(), 'native_reward': float(native_reward),
                     'native_score_high_water': float(self.env.env.current_score),
                     'native_task_cumulative': float(self.env.env.cumulative_rb),
                     'native_affect_cumulative': float(self.env.env.cumulative_ra),
                     'native_latest_hard_class': float(self.env.env.episode_arousal_trace[-1])
                         if self.env.env.episode_arousal_trace else None,
                     'game_observation': np.asarray(native_obs)[:81].tolist(),
                     'episode_end_reason': native_info.get('episode_end_reason'), 'schema': SCHEMA})
        self.score_model.assert_frozen()
        return obs, reward, terminated, truncated, info

EnvironmentAdapter=MatchedBaseline
print('Matched observation action and window adapter definitions/execution completed.')


Frozen increasing-preference scoring definitions/execution completed.
Task progress and fresh-pair rewards definitions/execution completed.
Matched observation action and window adapter definitions/execution completed.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
state=BaselineState(lambda p:(.6,{}),'MAX'); obs=state.reset(np.zeros(81));
for i in range(30): result=state.step(np.zeros(81),0)
print('Observation:',obs.shape,obs.dtype,'first pair:',result[4]['pair_id'],'reward:',result[1])

Observation: (86,) float32 first pair: [1, 1, 2] reward: 0.6
